# NCBI Multi-Omics Sequence Model — Kaggle free-tier run

**Purpose:** train a compact masked-sequence encoder on DNA, RNA and protein FASTA sequences from official NCBI RefSeq genome packages.

**Scope:** representative reference assemblies across Bacteria, Archaea, Fungi, Plants, Insecta and Nematoda. This is a reproducible multi-kingdom pilot, not the complete NCBI databases or every living organism; the full corpus exceeds Kaggle's free storage/compute.

**Source:** NCBI Datasets CLI can deliver genome, transcript/RNA and protein sequences in one package. Docs: https://www.ncbi.nlm.nih.gov/datasets/docs/v2/reference-docs/command-line/datasets/download/ and https://www.ncbi.nlm.nih.gov/datasets/docs/v2/how-tos/genomes/download-genome/

**Guardrails:** eight assemblies, 12,000 windows per accession/modality, small 4-layer Transformer, three epochs, no paid APIs, no automatic Hub upload. This is not a diagnostic or clinical model.

In [ ]:
!curl -fsSL https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/LATEST/linux-amd64/datasets -o /usr/local/bin/datasets
!chmod +x /usr/local/bin/datasets
!datasets version
!pip -q install tqdm

In [ ]:
import os,re,json,time,hashlib,zipfile,pathlib,subprocess,random,collections,io
from datetime import datetime,timezone
import torch,torch.nn as nn
from torch.utils.data import Dataset,DataLoader
from tqdm.auto import tqdm
SEED=42; random.seed(SEED); torch.manual_seed(SEED)
if not torch.cuda.is_available(): raise RuntimeError("Select a Kaggle GPU accelerator first.")
print(torch.cuda.get_device_name(0))
WORK=pathlib.Path("/kaggle/working/ncbi_multiomics"); WORK.mkdir(parents=True,exist_ok=True)
ACCESSIONS={
 "GCF_000005845.2":"Escherichia coli K-12 (Bacteria)",
 "GCF_000009045.1":"Bacillus subtilis 168 (Bacteria)",
 "GCF_000195955.2":"Mycobacterium tuberculosis H37Rv (Bacteria)",
 "GCF_000091665.1":"Methanocaldococcus jannaschii (Archaea)",
 "GCF_000146045.2":"Saccharomyces cerevisiae S288C (Fungi)",
 "GCF_000001735.4":"Arabidopsis thaliana TAIR10 (Plants)",
 "GCF_000001215.4":"Drosophila melanogaster (Insecta)",
 "GCF_000002985.6":"Caenorhabditis elegans (Nematoda)"
}
MAX_WINDOWS=12000; MAX_LEN=256

## 1. Download RefSeq genome packages

The accession list is explicit and auditable. Each package requests genomic DNA, RNA and proteins. Some assemblies may not have every annotation type; the notebook records which files were actually present.

In [ ]:
download_dir=WORK/"downloads"; download_dir.mkdir(exist_ok=True); downloaded=[]
for acc,organism in ACCESSIONS.items():
    out=download_dir/f"{acc}.zip"
    cmd=["datasets","download","genome","accession",acc,"--include","genome,rna,protein","--filename",str(out),"--no-progressbar"]
    print("Downloading",acc,organism)
    p=subprocess.run(cmd,capture_output=True,text=True)
    if p.returncode==0 and out.exists() and out.stat().st_size:
        downloaded.append({"accession":acc,"organism":organism,"zip":str(out),"bytes":out.stat().st_size})
    else: print("SKIP",acc,(p.stderr or p.stdout)[-1000:])
    time.sleep(0.25)
if len(downloaded)<4: raise RuntimeError(f"Only {len(downloaded)} assemblies downloaded; inspect network/CLI before training.")
print("Packages:",len(downloaded))

## 2. Extract DNA, RNA and protein sequence windows

Sequences are deduplicated and split by assembly accession, so windows from a given assembly cannot leak into both train and test.

In [ ]:
def fasta(lines):
    header=None; parts=[]
    for line in lines:
        line=line.strip()
        if not line: continue
        if line.startswith(">"):
            if header is not None: yield header,"".join(parts)
            header=line[1:]; parts=[]
        else: parts.append(line.upper())
    if header is not None: yield header,"".join(parts)
def modality(name):
    n=name.lower()
    if "protein" in n or n.endswith(".faa"): return "PROTEIN"
    if "rna" in n or "transcript" in n: return "RNA"
    if "genomic" in n or n.endswith(".fna"): return "DNA"
    return None
records=[]; file_manifest=[]
for pkg in downloaded:
    with zipfile.ZipFile(pkg["zip"]) as z:
        for name in z.namelist():
            mod=modality(name)
            if not mod or not name.lower().endswith((".fna",".faa",".fa",".fasta")): continue
            count=0
            with z.open(name) as raw:
                for header,seq in fasta(io.TextIOWrapper(raw,encoding="utf-8",errors="ignore")):
                    seq=re.sub("[^A-Z*]","",seq)
                    if len(seq)<64: continue
                    for start in range(0,len(seq)-MAX_LEN+1,MAX_LEN):
                        window=seq[start:start+MAX_LEN]
                        if len(window)==MAX_LEN:
                            records.append({"accession":pkg["accession"],"organism":pkg["organism"],"modality":mod,"sequence":window}); count+=1
                            if count>=MAX_WINDOWS: break
                    if count>=MAX_WINDOWS: break
            file_manifest.append({"accession":pkg["accession"],"file":name,"modality":mod,"windows":count})
records=list({(r["accession"],r["modality"],r["sequence"]):r for r in records}.values())
if not records: raise RuntimeError("No sequence windows extracted; inspect NCBI package layout.")
print("Windows:",len(records),collections.Counter(r["modality"] for r in records))
(WORK/"source_files.json").write_text(json.dumps({"assemblies":downloaded,"files":file_manifest,"windows":len(records)},indent=2))

## 3. Train a modality-aware masked Transformer encoder

The model shares residue-level parameters while receiving an explicit DNA/RNA/protein token. It is a masked-language-model encoder, not a sequence generator.

In [ ]:
SPECIAL={"[PAD]":0,"[MASK]":1,"[CLS]":2,"[SEP]":3,"[DNA]":4,"[RNA]":5,"[PROTEIN]":6}
SYMBOLS=list("ACGTUNRYKMSWBDHVXDEFGHILPQZJ*")
vocab={**SPECIAL,**{c:i+len(SPECIAL) for i,c in enumerate(SYMBOLS)}}
PAD=vocab["[PAD]"]; MASK=vocab["[MASK]"]; CLS=vocab["[CLS]"]; SEP=vocab["[SEP]"]
MOD={"DNA":vocab["[DNA]"],"RNA":vocab["[RNA]"],"PROTEIN":vocab["[PROTEIN]"]}
def encode(r): return [CLS,MOD[r["modality"]]]+[vocab.get(c,vocab["X"]) for c in r["sequence"][:MAX_LEN]]+[SEP]
assemblies=sorted({r["accession"] for r in records})
test_acc=set(assemblies[::5]); val_acc=set(assemblies[1::5]); train_acc=set(assemblies)-test_acc-val_acc
train_rows=[r for r in records if r["accession"] in train_acc]
val_rows=[r for r in records if r["accession"] in val_acc]
test_rows=[r for r in records if r["accession"] in test_acc]
if not train_rows or not val_rows or not test_rows: raise RuntimeError("Assembly-level split empty; add accessions.")
class SeqDS(Dataset):
    def __init__(self,rows): self.rows=rows
    def __len__(self): return len(self.rows)
    def __getitem__(self,i): return {"ids":encode(self.rows[i]),"mod":self.rows[i]["modality"]}
def collate(batch):
    x=torch.full((len(batch),MAX_LEN+3),PAD,dtype=torch.long); y=torch.full_like(x,-100); mods=[]
    for i,b in enumerate(batch):
        ids=b["ids"]; x[i,:len(ids)]=torch.tensor(ids); mods.append(b["mod"])
        pos=torch.arange(2,len(ids)-1); choose=pos[torch.rand(len(pos))<0.15]
        if len(choose):
            y[i,choose]=x[i,choose]; probs=torch.rand(len(choose)); x[i,choose[probs<0.8]]=MASK
            rand=(probs>=0.8)&(probs<0.9)
            if rand.any(): x[i,choose[rand]]=torch.randint(len(SPECIAL),len(vocab),(int(rand.sum()),))
    return x,y,mods
class MultiOmicsMLM(nn.Module):
    def __init__(self,vsize,d=256,layers=4,heads=8,maxlen=MAX_LEN+3):
        super().__init__(); self.tok=nn.Embedding(vsize,d,padding_idx=PAD); self.pos=nn.Embedding(maxlen,d)
        layer=nn.TransformerEncoderLayer(d_model=d,nhead=heads,dim_feedforward=4*d,dropout=.1,batch_first=True,norm_first=True,activation="gelu")
        self.encoder=nn.TransformerEncoder(layer,num_layers=layers,enable_nested_tensor=False); self.norm=nn.LayerNorm(d); self.head=nn.Linear(d,vsize)
    def forward(self,x):
        pos=torch.arange(x.shape[1],device=x.device).unsqueeze(0)
        h=self.tok(x)+self.pos(pos); return self.head(self.norm(self.encoder(h,src_key_padding_mask=(x==PAD))))
BATCH=64; device="cuda"; model=MultiOmicsMLM(len(vocab)).to(device)
train_loader=DataLoader(SeqDS(train_rows),batch_size=BATCH,shuffle=True,collate_fn=collate,num_workers=2)
val_loader=DataLoader(SeqDS(val_rows),batch_size=BATCH,shuffle=False,collate_fn=collate,num_workers=2)
test_loader=DataLoader(SeqDS(test_rows),batch_size=BATCH,shuffle=False,collate_fn=collate,num_workers=2)
optimizer=torch.optim.AdamW(model.parameters(),lr=3e-4,weight_decay=.01)
scaler=torch.amp.GradScaler("cuda")
print("Parameters:",sum(p.numel() for p in model.parameters()),"splits:",len(train_rows),len(val_rows),len(test_rows))

In [ ]:
def run_epoch(loader,training=False):
    model.train(training); loss_sum=0; n_tokens=0; correct=0; per=collections.defaultdict(lambda:[0,0])
    for x,y,mods in tqdm(loader,leave=False):
        x=x.to(device); y=y.to(device)
        if training: optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            with torch.autocast("cuda",dtype=torch.float16):
                logits=model(x); loss=nn.functional.cross_entropy(logits.reshape(-1,len(vocab)),y.reshape(-1),ignore_index=-100)
            if training:
                scaler.scale(loss).backward(); scaler.unscale_(optimizer); nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(optimizer); scaler.update()
        valid=y!=-100; n=int(valid.sum()); loss_sum+=float(loss.detach())*n; n_tokens+=n
        pred=logits.argmax(-1); correct+=(pred[valid]==y[valid]).sum().item()
        for i,m in enumerate(mods):
            mask=valid[i]; per[m][0]+=(pred[i][mask]==y[i][mask]).sum().item(); per[m][1]+=int(mask.sum())
    return {"loss":loss_sum/max(n_tokens,1),"masked_accuracy":correct/max(n_tokens,1),"tokens":n_tokens,"per_modality":{k:{"accuracy":v[0]/max(v[1],1),"tokens":v[1]} for k,v in per.items()}}
history=[]
for epoch in range(3):
    tr=run_epoch(train_loader,True); va=run_epoch(val_loader,False)
    history.append({"epoch":epoch+1,"train":tr,"validation":va}); print(history[-1])
test_metrics=run_epoch(test_loader,False)
OUT=WORK/"checkpoint"; OUT.mkdir(exist_ok=True)
torch.save(model.state_dict(),OUT/"multiomics_mlm.pt")
(OUT/"vocab.json").write_text(json.dumps(vocab,indent=2))
config={"architecture":"MultiOmicsMLM","vocab_size":len(vocab),"hidden_size":256,"layers":4,"heads":8,"max_len":MAX_LEN+3,"modalities":["DNA","RNA","PROTEIN"],"objective":"masked residue prediction"}
(OUT/"config.json").write_text(json.dumps(config,indent=2))

## 4. Save metrics, provenance and model hashes

Report held-out masked-residue loss and accuracy by modality. This is not evidence of biological function prediction, pathogenicity, clinical validity, or universal species coverage.

In [ ]:
def sha(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""): h.update(b)
    return h.hexdigest()
hashes={p.name:sha(p) for p in OUT.iterdir() if p.is_file()}
manifest={"model_name":"NCBI-MultiOmics-MLM-Pilot","created_utc":datetime.now(timezone.utc).isoformat(),"source":"NCBI RefSeq via NCBI Datasets CLI","assemblies":downloaded,"source_files":file_manifest,"windows":len(records),"train_assemblies":sorted(train_acc),"validation_assemblies":sorted(val_acc),"test_assemblies":sorted(test_acc),"history":history,"test_metrics":test_metrics,"config":config,"artifact_sha256":hashes,"status":"research_pilot_not_clinically_validated","limitations":["Representative subset, not full NCBI corpus","No clinical/diagnostic validation","Small number of held-out assemblies","RNA/protein availability depends on annotation"]}
(WORK/"training_manifest.json").write_text(json.dumps(manifest,indent=2,default=str))
print(json.dumps({"test_metrics":test_metrics,"checkpoint":str(OUT),"manifest":str(WORK/"training_manifest.json")},indent=2))

In [ ]:
import zipfile
BUNDLE=WORK/'ncbi_multiomics_model_artifacts.zip'
with zipfile.ZipFile(BUNDLE,'w',zipfile.ZIP_DEFLATED) as z:
    for path in [OUT/'multiomics_mlm.pt',OUT/'vocab.json',OUT/'config.json',WORK/'training_manifest.json',WORK/'source_files.json']:
        z.write(path,arcname=path.relative_to(WORK))
print('Artifact bundle:',BUNDLE,'bytes:',BUNDLE.stat().st_size)
